# NDJF Manual Verification and Position QA

This notebook adds peak-position diagnostics, simple candidate intensity metrics, a manual-review scaffold, and batch quicklook plots for visual event verification.

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import pandas as pd

from jpcz_catalog.analysis import (
    add_candidate_intensity_metrics,
    add_position_group_columns,
    annotate_events_with_peak_position,
    build_manual_verification_scaffold,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.verification import render_manual_verification_summary, write_text_summary

CATALOG_PATH = Path("outputs/verification/jpcz_catalog_ndjf.csv")
AUTO_CATALOG_PATH = Path("outputs/verification/jpcz_catalog_ndjf_position_intensity.csv")
SCAFFOLD_PATH = Path("outputs/verification/jpcz_catalog_ndjf_manual_verification.csv")
SUMMARY_PATH = Path("outputs/verification/ndjf_manual_verification_summary.md")
PLOT_DIR = Path("outputs/verification/ndjf_event_quicklooks")
CLOUD_VARIABLE = None

catalog_df = pd.read_csv(
    CATALOG_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)
ds = open_arco_era5()

if CLOUD_VARIABLE is not None and CLOUD_VARIABLE not in ds.data_vars:
    print(f"Requested cloud variable '{CLOUD_VARIABLE}' is not available in this ERA5 store. Falling back to wind + convergence only.")
    CLOUD_VARIABLE = None

augmented_catalog = annotate_events_with_peak_position(ds, catalog_df)
augmented_catalog = add_candidate_intensity_metrics(augmented_catalog)
augmented_catalog = add_position_group_columns(augmented_catalog)
augmented_catalog.to_csv(AUTO_CATALOG_PATH, index=False)

verification_scaffold = build_manual_verification_scaffold(augmented_catalog)
verification_scaffold.to_csv(SCAFFOLD_PATH, index=False)

summary_text = render_manual_verification_summary(
    total_events=len(verification_scaffold),
    auto_catalog_name=AUTO_CATALOG_PATH.name,
    scaffold_name=SCAFFOLD_PATH.name,
    plot_dir_name=PLOT_DIR.name,
    cloud_variable=CLOUD_VARIABLE,
)
summary_path = write_text_summary(SUMMARY_PATH, summary_text)

if PERSIST_OUTPUTS_TO_DRIVE:
    for path in [AUTO_CATALOG_PATH, SCAFFOLD_PATH, summary_path]:
        shutil.copy2(path, Path(DRIVE_OUTPUT_DIR) / path.name)

print(summary_text)
verification_scaffold.head()


In [ ]:
from pathlib import Path
import shutil

import pandas as pd

from jpcz_catalog.analysis import load_event_peak_snapshot
from jpcz_catalog.config import JPCZ_POLYGON_VERTICES, VORTICITY_BOX, WORKING_DOMAIN
from jpcz_catalog.detect import compute_divergence_field, prepare_detection_geometry
from jpcz_catalog.plotting import plot_event_peak_quicklook

PLOT_START = 0
PLOT_STOP = 12
OVERWRITE_EXISTING_PLOTS = False

if "verification_scaffold" not in globals():
    verification_scaffold = pd.read_csv(
        SCAFFOLD_PATH,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
if "ds" not in globals():
    ds = open_arco_era5()

PLOT_DIR.mkdir(parents=True, exist_ok=True)
drive_plot_dir = Path(DRIVE_OUTPUT_DIR) / "ndjf_event_quicklooks"
if PERSIST_OUTPUTS_TO_DRIVE:
    drive_plot_dir.mkdir(parents=True, exist_ok=True)

selected_events = verification_scaffold.iloc[PLOT_START:PLOT_STOP].copy()
geometry = None

for idx, row in selected_events.iterrows():
    peak_snapshot = load_event_peak_snapshot(
        ds,
        row["event_peak"],
        domain=WORKING_DOMAIN,
        cloud_variable=CLOUD_VARIABLE,
    )

    if geometry is None:
        geometry = prepare_detection_geometry(
            peak_snapshot.longitude,
            peak_snapshot.latitude,
            JPCZ_POLYGON_VERTICES,
        )

    divergence_field = compute_divergence_field(
        peak_snapshot,
        dx=geometry.dx,
        dy=geometry.dy,
    )

    cloud_field = peak_snapshot[CLOUD_VARIABLE] if CLOUD_VARIABLE is not None else None
    out_name = f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{idx:03d}.png"
    out_path = PLOT_DIR / out_name
    if out_path.exists() and not OVERWRITE_EXISTING_PLOTS:
        print(f"Skipping existing plot: {out_name}")
        continue

    plot_event_peak_quicklook(
        peak_snapshot,
        divergence_field,
        domain=WORKING_DOMAIN,
        polygon_vertices=JPCZ_POLYGON_VERTICES,
        vorticity_box=VORTICITY_BOX,
        title=(
            f"NDJF event {idx} | peak {pd.Timestamp(row['event_peak']).strftime('%Y-%m-%d %H:%M UTC')}\n"
            f"peak D={row['event_peak_D_1e5_s-1']:.2f} | duration={int(row['duration_hours'])} h | auto lat group={row['lat_position_group_auto']}"
        ),
        cloud_field=cloud_field,
        cloud_label=CLOUD_VARIABLE or "Cloud proxy",
        max_location=(row["peak_max_convergence_lat"], row["peak_max_convergence_lon"]),
        centroid_location=(row["peak_convergence_centroid_lat"], row["peak_convergence_centroid_lon"]),
        save_path=out_path,
    )

    if PERSIST_OUTPUTS_TO_DRIVE:
        shutil.copy2(out_path, drive_plot_dir / out_path.name)

    print(f"Saved {out_path}")


In [ ]:
preview_columns = [
    "event_peak",
    "event_peak_D_1e5_s-1",
    "duration_hours",
    "peak_max_convergence_lat",
    "peak_max_convergence_lon",
    "peak_convergence_centroid_lat",
    "peak_convergence_centroid_lon",
    "lat_position_group_auto",
    "lon_position_group_auto",
    "candidate_peak_convergence_1e5_s-1",
    "candidate_duration_weighted_convergence",
    "candidate_positive_box_vorticity_1e5_s-1",
    "candidate_peak_duration_vorticity_index",
    "verified_event",
    "cloud_band_present",
    "position_group_manual",
]

verification_scaffold.loc[PLOT_START:PLOT_STOP - 1, preview_columns]
